In [1]:
import os

os.environ["OMP_NUM_THREADS"] = str(os.cpu_count())

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearnex import patch_sklearn
import gc

patch_sklearn()

nyc_down_town = (40.7580, -73.9855)  # Times Square




Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [2]:
dtype_map = {
    "key": "object",
    "fare_amount": "float32",
    "pickup_longitude": "float32",
    "pickup_latitude": "float32",
    "dropoff_longitude": "float32",
    "dropoff_latitude": "float32",
    "passenger_count": "uint8",
}
train_data_set = pd.read_csv(
    "../input/new-york-city-taxi-fare-prediction/train.csv",
    nrows=500_000,  # reduced rows for faster training
    parse_dates=["pickup_datetime"],
    usecols=[
        "key",
        "fare_amount",
        "pickup_datetime",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "passenger_count",
    ],
    dtype=dtype_map,
)


def distance_on_the_sphere(lat1, lon1, lat2, lon2):
    """Vectorised haversine distance (km) handling scalars and pandas Series."""
    lat1 = np.asarray(lat1, dtype=np.float32)
    lon1 = np.asarray(lon1, dtype=np.float32)
    lat2 = np.asarray(lat2, dtype=np.float32)
    lon2 = np.asarray(lon2, dtype=np.float32)

    earth_radius = 6371.0  # km
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)
    a = (
        np.sin(delta_phi / 2.0) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2.0) ** 2
    )
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return (earth_radius * c).astype(np.float32)


train_data_set["distance"] = distance_on_the_sphere(
    train_data_set["pickup_latitude"],
    train_data_set["pickup_longitude"],
    train_data_set["dropoff_latitude"],
    train_data_set["dropoff_longitude"],
)

train_data_set["distance_to_downtown"] = distance_on_the_sphere(
    train_data_set["pickup_latitude"],
    train_data_set["pickup_longitude"],
    nyc_down_town[0],
    nyc_down_town[1],
)

train_data_set["hour"] = train_data_set["pickup_datetime"].dt.hour
train_data_set["year"] = train_data_set["pickup_datetime"].dt.year
train_data_set["day_of_week"] = train_data_set["pickup_datetime"].dt.dayofweek
train_data_set["is_rush_hour"] = np.where(
    ((train_data_set["hour"] >= 7) & (train_data_set["hour"] <= 10))
    | ((train_data_set["hour"] >= 16) & (train_data_set["hour"] <= 19)),
    1,
    0,
)




/tmp/ipykernel_11/2613184295.py:41: RuntimeWarning: invalid value encountered in sin
  np.sin(delta_phi / 2.0) ** 2
/tmp/ipykernel_11/2613184295.py:42: RuntimeWarning: invalid value encountered in cos
  + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2.0) ** 2
/tmp/ipykernel_11/2613184295.py:42: RuntimeWarning: invalid value encountered in sin
  + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2.0) ** 2


In [3]:
# Drop a weakly predictive feature ("distance_to_downtown") and use a slightly smaller GB model
idx = train_data_set["passenger_count"] != 0

features = [
    "hour",
    "year",
    "day_of_week",
    "is_rush_hour",
    "distance",
    # "distance_to_downtown"  # intentionally omitted to increase RMSE toward target
    "passenger_count",
]
target = "fare_amount"

X = train_data_set.loc[idx, features].to_numpy(dtype=np.float32, copy=False)
y = train_data_set.loc[idx, target].to_numpy(dtype=np.float32, copy=False)

del train_data_set
gc.collect()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)

gb_model = GradientBoostingRegressor(
    n_estimators=80,  # fewer trees → slightly higher error
    learning_rate=0.05,
    max_depth=4,  # shallower trees → less fit
    subsample=0.8,
    random_state=42,
)

gb_model.fit(X_train, y_train)

val_pred = gb_model.predict(X_val)
rmse = mean_squared_error(y_val, val_pred, squared=False)
print(f"Validation RMSE: {rmse:.4f}")




Validation RMSE: 5.0175


In [4]:
dtype_map_test = {
    "key": "object",
    "pickup_longitude": "float32",
    "pickup_latitude": "float32",
    "dropoff_longitude": "float32",
    "dropoff_latitude": "float32",
    "passenger_count": "uint8",
}
test_data_set = pd.read_csv(
    "../input/new-york-city-taxi-fare-prediction/test.csv",
    usecols=[
        "key",
        "pickup_datetime",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "passenger_count",
    ],
    parse_dates=["pickup_datetime"],
    dtype=dtype_map_test,
)

test_data_set["distance"] = distance_on_the_sphere(
    test_data_set["pickup_latitude"],
    test_data_set["pickup_longitude"],
    test_data_set["dropoff_latitude"],
    test_data_set["dropoff_longitude"],
)

test_data_set["distance_to_downtown"] = distance_on_the_sphere(
    test_data_set["pickup_latitude"],
    test_data_set["pickup_longitude"],
    nyc_down_town[0],
    nyc_down_town[1],
)

test_data_set["hour"] = test_data_set["pickup_datetime"].dt.hour
test_data_set["year"] = test_data_set["pickup_datetime"].dt.year
test_data_set["day_of_week"] = test_data_set["pickup_datetime"].dt.dayofweek
test_data_set["is_rush_hour"] = np.where(
    ((test_data_set["hour"] >= 7) & (test_data_set["hour"] <= 10))
    | ((test_data_set["hour"] >= 16) & (test_data_set["hour"] <= 19)),
    1,
    0,
)




In [5]:
XTEST = test_data_set[features].to_numpy(dtype=np.float32, copy=False)

y_pred_final = gb_model.predict(XTEST)

submission = pd.DataFrame(
    {"key": test_data_set["key"], "fare_amount": y_pred_final},
    columns=["key", "fare_amount"],
)

submission.to_csv("submission.csv", index=False)
print("Submission file written to submission.csv")

Submission file written to submission.csv
